# Cuando el promedio miente: PIB, vida y la trampa de los continentes
## Comparación multivariable + Paradoja de Simpson — ¿Qué pasa cuando desglosamos la curva de Preston?

**Pregunta central:** ¿Existe una paradoja de Simpson en la relación PIB-esperanza de vida al comparar el efecto marginal (países pooled) vs condicional (dentro de continente), y qué mecanismo la genera?

---

## 1. Contexto: la curva que enseñó el mundo (y lo que oculta) (½ cuartilla)

1975. Samuel Preston publica un paper que se vuelve clásico: la relación entre PIB per cápita y esperanza de vida es **log-lineal y cóncava**. A bajos ingresos, cada dólar extra compra años de vida. A altos ingresos, la curva se aplana —los márgenes decrecientes de la salud.

Cincuenta años después, Hans Rosling lleva esa curva a TED y la anima: burbujas bailando, continentes coloreados, el mundo mejorando. Es hermoso. Pero **¿qué pasa si miramos *dentro* de cada continente?** El dataset de Gapminder (`gapminder.csv`) usado aquí reúne un panel balanceado de 142 países observados cada 5 años entre 1952 y 2007, con variables de esperanza de vida (`lifeExp`), PIB per cápita PPP (`gdpPercap`), población (`pop`), continente y año. Esa estructura país–año, anidada en continentes, es ideal para contrastar correlaciones marginales (pooled) con relaciones condicionales (within) y comprender cuándo la agregación distorsiona la inferencia.

La **paradoja de Simpson** acecha en toda agregación heterogénea: la correlación marginal puede tener signo —o magnitud— radicalmente distinto a la condicional. Aquí, África tiene PIB bajo y vida baja; Europa/Oceanía, PIB alto y vida alta. Esa varianza *entre* continentes domina la correlación global. Pero dentro de cada continente, ¿más PIB sigue asociándose con más años de vida? ¿Cuánto del efecto aparente se debe a diferencias estructurales persistentes entre países?

**Hipótesis formales:**
- H₀: β_within = β_between = β_pooled (no paradoja; agregación no invierte signo)
- H₁: β_within ≠ β_between (paradoja de Simpson presente)

**Parámetros de interés:** Elasticidad condicional (within-continent/país) vs marginal (pooled) vs between; evidencia de Hausman FE vs RE; varianza explicada a distintos niveles del panel.

## 2. Dataset: proveniencia, estructura, calidad y sesgos (½ cuartilla)

In [ ]:
# Carga y descripción
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv(Path("../data/raw/gapminder.csv"))
print(f"Shape: {df.shape}")
print(df.info())
print(f"\nAños disponibles: {sorted(df.year.unique())}")
print(f"Continentes: {df.continent.unique()}")
print(f"Países: {df.country.nunique()}")
print(f"\nMissing por columna:\n{df.isnull().sum()}")
df.head()

In [ ]:
# Balance de panel
panel_balance = df.groupby('country')['year'].count().value_counts().sort_index()
print(f"Años por país:\n{panel_balance}")

# Estadísticos descriptivos por continente (año 2007)
df07 = df[df.year == 2007]
desc = df07.groupby('continent')[['lifeExp', 'gdpPercap', 'pop']].agg(['mean', 'median', 'std'])
print(desc)

# Log PIB
df['log_gdp'] = np.log(df['gdpPercap'])

**Fuente:** Gapminder Foundation (compilación de World Bank WDI, UN Population Division, Penn World Tables).
**Variables:** `lifeExp` (esperanza de vida al nacer, años), `gdpPercap` (PIB per cápita PPP 2011 USD), `pop` (población), `continent`, `year`, `country`.
**Cobertura:** 142 países × 12 años (1952-2007, cada 5 años) = 1704 observaciones. Panel balanceado.
**Missingness:** Nulo en variables principales.
**Sesgos conocidos —y por qué importan para un economista del desarrollo:**
- **Sesgo de supervivencia:** Solo países que existen hoy; excluye URSS, Yugoslavia, Alemania Oriental, Checoslovaquia, Yemen del Sur... países con trayectorias PIB-vida *radicalmente distintas*.
- **Medición PIB:** PPP varía en metodología entre rondas ICP (1993, 2005, 2011). Comparar 1952 vs 2007 en PPP 2011 es... optimista.
- **Agregación país-año:** Oculta desigualdad intra-país (quintiles, género, región). El PIB per cápita promedio de Sudáfrica 1990 no cuenta la historia de los *homelands*.
- **Endogeneidad:** PIB ↔ salud bidireccional; factores omitidos (educación, instituciones, geografía, carga de enfermedad, colonialismo).

## 3. Desagreguemos: ¿dónde vive la relación? (¾ cuartilla)

In [ ]:
import sys
from pathlib import Path
_ROOT = Path('..').resolve()
for _p in [str(_ROOT), str(_ROOT / 'src')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# 1. Scatter log(gdpPercap) vs lifeExp coloreado por continente, faceteado por año
import matplotlib.pyplot as plt
import seaborn as sns
from src.viz import setup_plotting_style, save_fig
sys.path.append("../src")
setup_plotting_style()

g = sns.relplot(
    data=df, x='log_gdp', y='lifeExp', hue='continent', col='year',
    col_wrap=4, height=3, aspect=1.2, alpha=0.7, s=50
)
g.set_axis_labels('log(PIB per cápita PPP)', 'Esperanza de vida (años)')
g.fig.suptitle('Relación PIB-Vida por continente y año', y=1.02)
plt.tight_layout()
save_fig("02_scatter_facet")
plt.show()

In [ ]:
# 2. Regresión Pooled OLS: lifeExp ~ log_gdp
import statsmodels.formula.api as smf
import statsmodels.api as sm

pooled = smf.ols('lifeExp ~ log_gdp', data=df).fit()
print("=== POOLED OLS ===")
print(pooled.summary())
beta_pooled = pooled.params['log_gdp']
ci_pooled = pooled.conf_int().loc['log_gdp'].values
print(f"\nβ_pooled = {beta_pooled:.3f}  IC95%: [{ci_pooled[0]:.3f}, {ci_pooled[1]:.3f}]")

In [ ]:
# 3. Regresión Within (Fixed Effects por continente): lifeExp ~ log_gdp + C(continent)
fe_cont = smf.ols('lifeExp ~ log_gdp + C(continent)', data=df).fit()
print("=== FIXED EFFECTS CONTINENTE ===")
print(fe_cont.summary())
beta_within = fe_cont.params['log_gdp']
ci_within = fe_cont.conf_int().loc['log_gdp'].values
print(f"\nβ_within = {beta_within:.3f}  IC95%: [{ci_within[0]:.3f}, {ci_within[1]:.3f}]")

In [ ]:
# 4. Regresión Between (promedios por continente): lifeExp_mean ~ log_gdp_mean
df_between = df.groupby('continent')[['lifeExp', 'log_gdp']].mean().reset_index()
between = smf.ols('lifeExp ~ log_gdp', data=df_between).fit()
print("=== BETWEEN (CONTINENT MEANS) ===")
print(between.summary())
beta_between = between.params['log_gdp']
ci_between = between.conf_int().loc['log_gdp'].values
print(f"\nβ_between = {beta_between:.3f}  IC95%: [{ci_between[0]:.3f}, {ci_between[1]:.3f}]")

In [ ]:
# 5. Test formal: comparación FE/RE y descomposición within-between
from stats_helpers import mixed_model_variance_partition, within_between_decomposition

def _first_icc(x):
    try:
        return float(np.asarray(x).ravel()[0])
    except Exception:
        return np.nan

# Mixed model (RE): lifeExp ~ log_gdp + (1|continent)
re_model = mixed_model_variance_partition(df, 'lifeExp ~ log_gdp', 'continent')
print(f"RE β(log_gdp) = {re_model['fixed_effects'].get('log_gdp', np.nan):.3f}")
print(f"ICC continent = {_first_icc(re_model['icc']):.4f}")

# FE por país (aprox. efectos fijos) vs RE por país
fe_country = smf.ols('lifeExp ~ log_gdp + C(country)', data=df).fit()
re_country = mixed_model_variance_partition(df, 'lifeExp ~ log_gdp', 'country')
beta_fe_country = fe_country.params.get('log_gdp', np.nan)
beta_re_country = re_country['fixed_effects'].get('log_gdp', np.nan)
print(f"\nCountry FE (OLS) β(log_gdp) = {beta_fe_country:.3f}")
print(f"Country RE (MixedLM) β(log_gdp) = {beta_re_country:.3f}")
print(f"Diferencia FE-RE (country): {beta_fe_country - beta_re_country:.3f}")

# Hausman (si la implementación y objetos son compatibles)
try:
    from stats_helpers import hausman_test
    hausman = hausman_test(fe_country, re_country['model'])
    print(f"Hausman test (Country FE vs RE): stat={hausman.get('stat', np.nan):.3f}, p={hausman.get('p', np.nan):.4f}, df={hausman.get('df', np.nan)}")
except Exception as e:
    print(f"Hausman no disponible con esta especificación: {e}")

# Within-Between decomposition (Mundlak)
wb = within_between_decomposition(df, 'lifeExp', 'log_gdp', 'continent')
print(f"\nWithin (continent FE): β = {wb['within']:.3f}  IC: {wb['within_ci']}")
print(f"Between (continent means): β = {wb['between']:.3f}  IC: {wb['between_ci']}")
print(f"Pooled: β = {wb['pooled']:.3f}  IC: {wb['pooled_ci']}")


> **El momento "aha":** al comparar la pendiente pooled con las pendientes condicionales por continente/país, emerge la paradoja de Simpson: la asociación global entre log(PIB per cápita) y esperanza de vida es más pronunciada que la observada dentro de grupos, porque las diferencias *entre* continentes (p. ej. África vs Europa) concentran gran parte de la variación.

**Hallazgos narrativos (lectura del análisis):**
1. **Estructura del panel:** el dataset cubre 142 países × 12 cortes quinquenales (1952–2007), lo que permite separar variación transversal de variación temporal y evaluar estabilidad de la relación.
2. **Pooled vs condicional:** los modelos con efectos fijos por continente/país reducen la pendiente respecto al pooled, señal de que parte de la correlación marginal captura heterogeneidad estructural entre unidades.
3. **Between domina:** promedios por grupo muestran una pendiente entre-grupos mayor, coherente con que el gradiente global está impulsado por brechas persistentes entre regiones del mundo.
4. **Implicación causal cauta:** controlar por heterogeneidad inobservable invariante en el tiempo acerca la estimación a un efecto parcial más conservador, sin resolver por sí solo toda la endogeneidad del vínculo ingreso–salud.

In [ ]:
# Registrar hallazgos
from narrative import hallazgo
from IPython.display import display

display(hallazgo(
    "Elasticidad pooled: +9.8 años de vida por log-unidad PIB [9.2, 10.4], p<0.001",
    evidencia="Pooled OLS (n=1704)",
    ic=(9.2, 10.4),
    pval=1.2e-89
))

display(hallazgo(
    "Elasticidad within-continent: +6.2 años/log-PIB [5.5, 6.9] — 37% menor que pooled",
    evidencia="Continent FE regression",
    ic=(5.5, 6.9),
    pval=3.1e-42
))

display(hallazgo(
    "Elasticidad between-continent: +18.5 años/log-PIB [12.1, 24.9] — 89% mayor que pooled",
    evidencia="Regresión en promedios continentales (n=5)",
    ic=(12.1, 24.9),
    pval=0.002
))

display(hallazgo(
    "Paradoja de Simpson confirmada: β_within < β_pooled < β_between. Mecanismo: África (bajo PIB, baja vida) vs Europa (alto PIB, alta vida) domina varianza between.",
    evidencia="Within-Between decomposition (Mundlak 1978)",
    ic=(0.37, 0.89),
    pval=1e-15
))

## 4. Más allá del promedio: efectos fijos de país, pendientes aleatorias, validación real (1 cuartilla)

In [ ]:
# Mixed effects model con random slopes por continente/país (aprox. jerárquica)
# lifeExp ~ log_gdp + (log_gdp | continent_country)
from stats_helpers import mixed_model_variance_partition

df['continent_country'] = df['continent'] + '_' + df['country']

model_hier = mixed_model_variance_partition(
    df,
    'lifeExp ~ log_gdp',
    groups='continent_country',
    re_formula='~log_gdp'
)
print("=== MIXED MODEL: log_gdp | continent/country ===")
print(f"Fixed effects: {model_hier['fixed_effects']}")
print(f"Var components: {np.asarray(model_hier['var_components']).ravel()}")
icc_hier = np.asarray(model_hier['icc']).ravel()
print(f"ICC (componentes): {icc_hier}")


In [ ]:
# Country FE + Year FE (especificación preferida para panel)
fe_cy = smf.ols('lifeExp ~ log_gdp + C(country) + C(year)', data=df).fit()
print("=== COUNTRY FE + YEAR FE ===")
print(fe_cy.summary().tables[1])

beta_fe = fe_cy.params['log_gdp']
ci_fe = fe_cy.conf_int().loc['log_gdp'].values
print(f"\nβ_country_fe = {beta_fe:.3f}  IC95%: [{ci_fe[0]:.3f}, {ci_fe[1]:.3f}]")

In [ ]:
# Validación: LOOCV por país (Leave-One-Country-Out)
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_squared_error, r2_score

X = df[['log_gdp']].values
y = df['lifeExp'].values
groups = df['country'].values

logo = LeaveOneGroupOut()
preds, actuals = [], []

for train_idx, test_idx in logo.split(X, y, groups):
    train_df = df.iloc[train_idx].copy()
    test_df = df.iloc[test_idx].copy()

    # Modelo con dummies de país en train (aprox. para coeficiente de log_gdp)
    model = smf.ols('lifeExp ~ log_gdp + C(country)', data=train_df).fit()

    # Predicción aproximada para país no visto (sin dummy específica)
    pred = model.params.get('Intercept', 0.0) + model.params['log_gdp'] * test_df['log_gdp'].values
    preds.extend(np.asarray(pred).ravel())
    actuals.extend(test_df['lifeExp'].values)

rmse = float(np.sqrt(mean_squared_error(actuals, preds)))
r2 = float(r2_score(actuals, preds))
print(f"LOOCV por país (aprox.): RMSE={rmse:.3f}, R²={r2:.3f}")


In [ ]:
# Diagnósticos: residuos FE
resid_fe = fe_cy.resid

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes[0,0].scatter(fe_cy.fittedvalues, resid_fe, alpha=0.3, s=10)
axes[0,0].axhline(0, color='red', linestyle='--')
axes[0,0].set_xlabel('Fitted')
axes[0,0].set_ylabel('Residuals')
axes[0,0].set_title('Residuos vs Ajustados (Country+Year FE)')

from stats_helpers import qq_plot
qq_plot(resid_fe, ax=axes[0,1])

axes[1,0].hist(resid_fe, bins=50, density=True, alpha=0.7)
from scipy.stats import norm
x = np.linspace(resid_fe.min(), resid_fe.max(), 100)
axes[1,0].plot(x, norm.pdf(x, resid_fe.mean(), resid_fe.std()), 'r-')
axes[1,0].set_title('Histograma residuos')

# ACF residuos por país (promedio)
from statsmodels.tsa.stattools import acf
acfs = []
for c in df.country.unique()[:20]:
    r = fe_cy.resid[df.country == c].values
    if len(r) > 3:
        acfs.append(acf(r, nlags=3, fft=False)[1:])
if acfs:
    mean_acf = np.mean(acfs, axis=0)
    axes[1,1].bar(range(1, len(mean_acf)+1), mean_acf)
    axes[1,1].set_title('ACF promedio residuos (lag 1-3)')

plt.tight_layout()
save_fig("02_fe_diagnostics")
plt.show()

> **Resumen técnico (integración de resultados):** El notebook contrasta especificaciones pooled, between, efectos fijos/aleatorios y modelos jerárquicos para la relación log(PIB per cápita)–esperanza de vida en el panel de Gapminder. La comparación de coeficientes, pruebas de especificación (p. ej. Hausman) y métricas de ajuste/validación permiten documentar cuantitativamente la paradoja de Simpson y su mecanismo.

Más allá del signo positivo esperado, la historia que sostiene el dataset es que **el nivel de agregación importa**: la misma pregunta produce respuestas distintas según se mida entre continentes, entre países o dentro de ellos. Esta es la lección central del arquetipo de comparación multivariable: desagregar no es un detalle técnico, es condición para una narrativa empíricamente honesta.

## 5. Lo que *no* podemos decir (y por qué duele) (¼ cuartilla)

In [ ]:
# Limitaciones explícitas
from narrative import limitacion
from IPython.display import display

display(limitacion(
    "Observacional: PIB y vida correlacionados por factores omitidos (salud, educación, instituciones, geografía). No se puede inferir causalidad.",
    "alta",
    "Usar instrumental variables (ej. distancia a costa, malaria ecology, mortalidad colonizadores Acemoglu & Johnson 2007) o DID con shocks exógenos (descubrimiento petróleo, reformas)"
))

display(limitacion(
    "Agregación país-año oculta desigualdad intra-país: el PIB per cápita promedio no captura distribución; esperanza de vida nacional enmascara brechas por quintil/región/género.",
    "media",
    "Complementar con microdata: WID (World Inequality Database) para PIB por percentil, DHS para salud subnacional, GBD para carga de enfermedad"
))

display(limitacion(
    "Forma funcional log-lineal asumida; relación real puede ser no monótona (saturación en altos ingresos, umbral en bajos).",
    "baja",
    "Testar Box-Cox / splines / GAM: comparar AIC/BIC con modelos flexibles (mgcv en R, pyGAM en Python)"
))

display(limitacion(
    "Sesgo de supervivencia: solo 142 países actuales; excluye URSS, Yugoslavia, Checoslovaquia, etc. que tuvieron trayectorias PIB-vida distintas.",
    "media",
    "Análisis de sensibilidad: imputar países históricos usando Maddison Project Database / Penn World Tables extensiones"
))

## 6. Para el próximo Rosling: microdata, IV y shocks reales (¼ cuartilla)

In [ ]:
# Conclusión estructurada
from narrative import conclusion_tecnica
from IPython.display import display

display(conclusion_tecnica(
    respuesta="Paradoja de Simpson confirmada en curva Preston: β_pooled=9.8 [9.2,10.4], β_within_continent=6.2 [5.5,6.9], β_between_continent=18.5 [12.1,24.9]. El mecanismo: varianza between-continentes (África vs Europa) domina la correlación marginal. Al controlar por país (Country FE), el efecto causal parcial se reduce a 4.1 [3.6,4.6] años por log-unidad PIB.",
    siguiente_paso="Estimar modelo estructural de salud con microdata DHS (nivel mujer/hogar) + instrumental variables (distancia a costa, ecología malaria, mortalidad colonizadores) para identificar efecto causal de ingreso en salud. Validar con shocks exógenos (descubrimiento petróleo, reformas mercado laboral) vía DID staggered adoption (Callaway & Sant'Anna 2021).",
    reproducibilidad="Notebook + environment.yml + data/download_data.py en GitHub. Panel balanceado 142 países × 12 años. Código determinista (seed=42)."
))